# Pipeline Test
Run each stage of the pipeline end-to-end: ingest → chunk → extract frames → caption.

In [1]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")

Repo root: /home/rhianna/Documents/Repos/DiveVisionApp


## 1. Ingestion
Scan raw videos and register them in `video_registry.json`.

In [2]:
from src.ingestion import register_video, load_registry
from src.constants import RAW_VIDEO_DIR, VIDEO_REGISTRY_JSON

video_extensions = {".mp4", ".MP4", ".mov", ".MOV"}
raw_videos = [p for p in RAW_VIDEO_DIR.iterdir() if p.suffix in video_extensions]
print(f"Found {len(raw_videos)} video(s) in {RAW_VIDEO_DIR}:")
for v in raw_videos:
    print(f"  {v.name}")

CalledProcessError: Command '['pass', 'show', 'APIs/hugging_face']' returned non-zero exit status 2.

In [ ]:
for video_path in raw_videos:
    print(f"Registering: {video_path.name}")
    register_video(video_path)
    print()

video_registry = load_registry()
print(f"Registry has {len(video_registry)} video(s). Saved to: {VIDEO_REGISTRY_JSON}")

In [ ]:
# Inspect a single video record
video_id, record = next(iter(video_registry.items()))
import pprint
pprint.pprint(record)

## 2. Chunking
Split each video into 20-second chunks and save to `chunk_registry.json`.

In [ ]:
from src.chunk_video import build_chunk_registry, save_chunk_registry, CHUNK_REGISTRY_JSON, CHUNK_DURATION_S

chunk_registry = build_chunk_registry()
save_chunk_registry(chunk_registry)

total_videos = len({c["video_id"] for c in chunk_registry.values()})
print(f"Chunked {total_videos} video(s) into {len(chunk_registry)} chunks ({CHUNK_DURATION_S}s each)")
print(f"Saved to: {CHUNK_REGISTRY_JSON}")

In [ ]:
# Inspect chunks per video
for chunk_id, chunk in chunk_registry.items():
    vid = video_registry[chunk["video_id"]]
    print(f"  {vid['filename']}  chunk {chunk['chunk_index']:03d}: {chunk['start_s']}s – {chunk['end_s']}s  ({chunk['duration_s']}s)")

## 3. Frame Extraction
Sample a frame every 3.33 seconds from each chunk and save as JPEGs.

In [ ]:
from src.extract_frames import extract_all_frames, SAMPLE_INTERVAL_S
from src.constants import FRAMES_DIR

frame_results = extract_all_frames()

total_frames = sum(len(paths) for paths in frame_results.values())
print(f"Extracted {total_frames} frames across {len(frame_results)} chunks (every {SAMPLE_INTERVAL_S}s)")
print(f"Saved under: {FRAMES_DIR}")

In [ ]:
# Preview extracted frames for one chunk
import matplotlib.pyplot as plt
from PIL import Image

sample_chunk_id = next(iter(frame_results))
sample_paths = sorted(frame_results[sample_chunk_id])

fig, axes = plt.subplots(1, len(sample_paths), figsize=(4 * len(sample_paths), 4))
if len(sample_paths) == 1:
    axes = [axes]
for ax, p in zip(axes, sample_paths):
    ax.imshow(Image.open(p))
    ax.set_title(p.name, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Captioning
Run BLIP on each frame and save captions to `caption_registry.json`.

In [ ]:
from src.caption_frames import load_model, caption_image, CAPTION_REGISTRY_JSON

processor, model = load_model()
print("Model loaded.")

In [ ]:
# Test captioning on a single frame before running the full pipeline
test_frame = sample_paths[0]
caption = caption_image(test_frame, processor, model)
print(f"{test_frame.name}: {caption}")

img = Image.open(test_frame)
plt.imshow(img)
plt.title(caption, wrap=True)
plt.axis("off")
plt.show()

In [ ]:
# Run captioning across all chunks (writes caption_registry.json)
from src.caption_frames import caption_all_chunks

caption_registry = caption_all_chunks()
print(f"\nCaption registry saved to: {CAPTION_REGISTRY_JSON}")

In [1]:
# Inspect the caption registry
import json
from src.constants import CAPTION_REGISTRY_JSON, VIDEO_REGISTRY_JSON, CHUNK_REGISTRY_JSON

import json                                                                   
                                                                            
# with open(CAPTION_REGISTRY_JSON, 'r') as f:                                             
#     caption_registry = json.load(f)
# with open(VIDEO_REGISTRY_JSON, 'r') as f:                                             
#     video_registry = json.load(f)
# with open(CHUNK_REGISTRY_JSON, 'r') as f:                                             
#     chunk_registry = json.load(f)

caption_data = json.loads(CAPTION_REGISTRY_JSON.read_text())
video_registry = json.loads(VIDEO_REGISTRY_JSON.read_text())
chunk_registry = json.loads(CHUNK_REGISTRY_JSON.read_text())

for chunk_id, entry in caption_data.items():
    vid = video_registry[entry["video_id"]]
    print(f"\n{vid['filename']} — chunk {entry['chunk_index']:03d}")
    for frame_name, cap in entry["captions"].items():
        print(f"  {frame_name}: {cap}")


GX014538.MP4 — chunk 000
  frame_0.000s.jpg: there is a man swimming in the ocean with a lot of fish
  frame_13.320s.jpg: there is a person swimming in the ocean with a scuba board
  frame_16.650s.jpg: there are two divers in the water near a coral reef
  frame_19.980s.jpg: there are two divers in the water with a variety of corals
  frame_3.330s.jpg: there is a large amount of fish swimming in the ocean
  frame_6.660s.jpg: there are many fish swimming in the ocean near a coral reef
  frame_9.990s.jpg: there is a fish that is swimming in the ocean

GX014538.MP4 — chunk 001
  frame_20.000s.jpg: there are two divers in the water with a variety of corals
  frame_23.330s.jpg: there are two divers in the water near a coral reef
  frame_26.660s.jpg: there are many different types of corals on the ocean floor
  frame_29.990s.jpg: there is a large group of fish swimming over a coral reef
  frame_33.320s.jpg: there is a large group of fish swimming on a coral reef

B666D060-459D-41A4-A148-3560

In [2]:
caption_data

{'5ff0c27e-b975-48a6-a1ce-67be92c02341': {'chunk_id': '5ff0c27e-b975-48a6-a1ce-67be92c02341',
  'video_id': '5cfb519d-46ad-4bfb-ad32-8317be28a0b1',
  'chunk_index': 0,
  'captions': {'frame_0.000s.jpg': 'there is a man swimming in the ocean with a lot of fish',
   'frame_13.320s.jpg': 'there is a person swimming in the ocean with a scuba board',
   'frame_16.650s.jpg': 'there are two divers in the water near a coral reef',
   'frame_19.980s.jpg': 'there are two divers in the water with a variety of corals',
   'frame_3.330s.jpg': 'there is a large amount of fish swimming in the ocean',
   'frame_6.660s.jpg': 'there are many fish swimming in the ocean near a coral reef',
   'frame_9.990s.jpg': 'there is a fish that is swimming in the ocean'}},
 '0e75bc9e-48cd-4148-863a-24ee8161612d': {'chunk_id': '0e75bc9e-48cd-4148-863a-24ee8161612d',
  'video_id': '5cfb519d-46ad-4bfb-ad32-8317be28a0b1',
  'chunk_index': 1,
  'captions': {'frame_20.000s.jpg': 'there are two divers in the water with a v